# Technische Prüfung: Bestand, Restvolumen, Datenlage

Das Gegenstück zu `01_dashboard.ipynb`. Dort steht, was Fachexperten sehen sollen;
hier steht, was zu prüfen ist, bevor man den Zahlen traut – und welche fachlichen
Fragen offen sind.

Umgesetzt ist Spec 5.1 (Restvolumen je Projekt). Die Monte-Carlo-Simulation aus 5.4
fehlt; warum, steht am Ende.

In [ ]:
# Nur in Google Colab: Projekt aus GitHub installieren, weil das lokale venv dort
# nicht zur Verfuegung steht. Lokal passiert hier nichts, dort liefert `uv sync` die
# Umgebung. Das Repository ist oeffentlich, deshalb braucht pip kein Token.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Fuer einen reproduzierbaren Lauf auf einen Tag setzen statt auf "main".
PAKET_REF = "main"
PAKET_URL = f"git+https://github.com/it-agile/umsatzprognose-clockodo.git@{PAKET_REF}"

if IN_COLAB:
    # Zwei Aufrufe, jeder aus einem eigenen Grund.
    #
    # Der erste beschafft die Abhaengigkeiten, und nur die fehlenden: Colab pinnt
    # pandas 2.2.3 (google-colab) und numpy < 2.3 (numba); "pandas>=2.2" ist damit
    # erfuellt, pip laesst beide stehen. Fuer plotly gilt dasselbe - "plotly>=5" ist
    # von Colabs mitgelieferter Version erfuellt.
    #
    # Der zweite erneuert ausschliesslich unseren Code. --force-reinstall ist noetig,
    # weil die Versionsnummer ueber Commits hinweg 0.1.0 bleibt und pip die
    # Anforderung sonst fuer erfuellt haelt - "pip install git+...@main" laesst einen
    # installierten Stand dann unangetastet, ohne Fehlermeldung (verifiziert am
    # 24.08.2026, der Import schlug danach mit ModuleNotFoundError fehl).
    # --no-deps haelt pandas und numpy aus dem Reinstall heraus: ohne dieses Flag zog
    # der Aufruf pandas 3.0.5 und numpy 2.5.2 nach und brach google-colab 1.0.0 und
    # numba 0.61.2.
    #
    # Nach einem neuen Push zusaetzlich die Runtime neu starten. Sonst bleibt das alte
    # Paket im Speicher, und ein neuer Name in einem alten Modul endet als ImportError.
    !pip install --quiet "$PAKET_URL"
    !pip install --quiet --force-reinstall --no-deps "$PAKET_URL"

# Diese Ausgabe erzeugt nur die aktuelle Fassung der Zelle. Fehlt sie, ist nicht das
# Paket alt, sondern das Notebook: pip erneuert die .ipynb nicht. Dann das Notebook
# neu laden ueber File -> Open notebook -> GitHub.
print("Installationszelle Technik, Stand 2026-08-24 | Colab:", IN_COLAB)

## Bestand laden

Ein Aufruf, sechs Abrufe: Kunden, Personen, Sollarbeitszeiten, Projekte, Verbrauch
samt Personenanteilen und Monatsumsätze. Die Eigenheiten der Endpunkte sind in
`umsatzprognose.clockodo` dokumentiert und dort an echten Antworten belegt.

In [ ]:
from umsatzprognose import BestandRepository, Dashboard

bestand = await BestandRepository.mit_automatischen_zugangsdaten().laden_async()
dashboard = Dashboard(bestand)

print(f"Stichtag: {bestand.stichtag}")
print(f"Projekte gesamt:    {len(bestand.projekte)}")
print(f"davon aktiv:        {len(bestand.aktive_projekte)}")
print(f"davon im Scope:     {len(bestand.im_prognose_scope)}  (aktiv und mit Euro-Budget)")
print(f"Personen:           {len(bestand.mitarbeiter)}")
print(f"Kunden mit Projekt: {len(bestand.kunden)}")

## Auftragsvolumen und Restvolumen (Spec 5.1)

`roh` ist `Budget − Verbrauch` und kann negativ sein, weil `budget.hard` in dieser
Installation `false` ist. Prognosewirksam ist `max(0, roh)`: eine Überschreitung kann
nur historisch entstehen, die Prognose überschreitet das Budget nicht (Spec 5.1). Die
Differenz beider Summen ist die Summe der Überschreitungen – ein Kalibrierungssignal,
kein Fehler.

In [ ]:
from umsatzprognose.domaene.zahlen import euro

roh = sum(p.restvolumen_roh or 0.0 for p in bestand.im_prognose_scope)
gesamt = bestand.restvolumen_prognosewirksam

print(f"Auftragsvolumen im Scope:    {euro(bestand.auftragsvolumen):>18}")
print(f"Restvolumen roh:             {euro(roh):>18}")
print(f"Restvolumen prognosewirksam: {euro(gesamt):>18}")
print(f"Summe der Überschreitungen:  {euro(gesamt - roh):>18}")

dashboard.projekttabelle()

## Aufteilungsschlüssel je Person (Spec 5.4, Schritt 3)

Der historische Anteil je Person an den Gesamtstunden des Projekts, aus der
Doppelgruppierung `projects_id` × `users_id`. Er wird unverändert in die Zukunft
fortgeschrieben – wechselt die Teambesetzung, veraltet er. Das ist laut Spec ein
Kalibrierungsthema und keine Modelländerung.

In [ ]:
for projekt in bestand.im_prognose_scope[:5]:
    anteile = projekt.anteil_je_mitarbeiter()
    groesste = sorted(anteile.items(), key=lambda paar: paar[1], reverse=True)[:3]
    verteilung = ", ".join(f"{person} {anteil:.0%}" for person, anteil in groesste)
    wort = "Person " if len(anteile) == 1 else "Personen"
    print(f"{projekt.bezeichnung[:48]:<48} {len(anteile):>2} {wort} | {verteilung}")

## Sollarbeitszeit (Spec 5.3)

**Hier weicht die Umsetzung von der Spec ab.** Abschnitt 4 nennt
`default_target_hours` aus `/v3/users`. Das Feld ist ein Boolean-Schalter, keine
Stundenzahl – geprüft an allen Personen der Installation. Die Sollarbeitszeit steht im
unversionierten `/targethours`, je Person mit Gültigkeitszeitraum und Stunden je
Wochentag.

Die **verfügbare** Kapazität aus 5.3 ist damit noch nicht berechnet: dafür fehlen die
geplanten Abwesenheiten (`/v4/absences`) und der Abschlag für ungeplante Abwesenheit.

In [ ]:
from umsatzprognose.domaene.zahlen import stunden as stunden_text

aktive = [m for m in bestand.mitarbeiter if m.aktiv]
ohne_sollzeit = [m for m in aktive if m.wochenstunden(bestand.stichtag) is None]
stunden = sum(m.wochenstunden(bestand.stichtag) or 0 for m in aktive)

print(f"Aktive Personen: {len(aktive)}, ohne hinterlegte Sollzeit: {len(ohne_sollzeit)}")
print(f"Vereinbarte Wochenstunden gesamt: {stunden_text(stunden)}")

## Datenlage und offene fachliche Fragen

In [ ]:
dashboard.hinweise()

### ENTSCHEIDEN – aktive Projekte ohne Budget

Sie fallen aus der Prognose, weil ihnen ein bezifferbares Auftragsvolumen fehlt. Ein
Blick auf die Liste unten zeigt, was das überwiegend ist: Schulungs- und
Ausbildungsprodukte aus dem offenen Kursangebot, also Katalogpositionen ohne
beauftragtes Volumen. Genau das rechnet die Spec dem
Kurzfristgeschäft zu und schließt es aus dem MVP aus. Zu prüfen bleibt, ob darunter
echte Bestandsprojekte stecken, bei denen nur das Budget fehlt – dann ist es ein
Pflegethema, kein Modellthema.

In [ ]:
ohne_budget = [p for p in bestand.aktive_projekte if not p.budget.verwertbar]
print(f"{len(ohne_budget)} aktive Projekte ohne verwertbares Budget\n")
for projekt in ohne_budget[:12]:
    grund = projekt.budget.sonderfall or "kein Budget gesetzt"
    print(f"  {projekt.bezeichnung[:58]:<58} {grund}")

### Aktive Projekte, die als abgeschlossen markiert sind

Sie fallen aus der Prognose: `completed` schlägt `active` (Spec 5.0). Spec 7 hält
`completed` für ein zuverlässiges Endesignal, während `active` auch ein nicht
nachgezogener Schalter sein kann – und offenes Restvolumen eines beendeten Projekts wird
nicht mehr abgerufen. Die Liste bleibt sichtbar, damit die Regel prüfbar ist und die
Projekte nicht still verschwinden.

In [ ]:
beendet = [p for p in bestand.aktive_projekte if p.abgeschlossen]
for projekt in beendet:
    offen = projekt.restvolumen_prognosewirksam
    betrag = "kein Budget" if offen is None else euro(offen)
    print(f"  {projekt.bezeichnung[:58]:<58} offen: {betrag}")

## Was für die Simulation (Spec 5.4) noch fehlt

**Effektiver Stundensatz und Pauschalleistungen – entschieden.** `hourly_rate` aus
`/v2/entrygroups` taugt nicht: nur bei einer Minderheit der Gruppen gesetzt und dort
meist 0. Der Satz wird aus Umsatz und Zeit abgeleitet; Pauschalleistungen mit gebuchter Zeit sind
darin normalisiert enthalten (Spec 5.1). Offen bleibt allein der Fall Umsatz **ohne**
erfasste Zeit: diese Projekte gehen mit ihrem Restvolumen ein, verbrauchen aber keine
Kapazität, und werden als Hinweis ausgewiesen.

**Abrufquote-Verteilung (5.2) – Form steht, Zahlen fehlen.** Empirische Verteilung über
Projekt-Monate: Quote = Verbrauch im Monat geteilt durch Restvolumen zu Monatsbeginn,
gezogen mit Zurücklegen. Datenquelle ist `/v2/entrygroups` mit
`grouping[]=projects_id&grouping[]=month`. Geschätzt ist sie noch nicht.

**Bereits gebuchte Beträge im Horizont (5.4) – Untergrenze, nicht Verbrauch.** Das
Verbrauchsfenster endet am Stichtag; was später datiert ist, liegt im Horizont. Der Fall
tritt regelmäßig auf, gemessen an der Prognosesumme aber in kleiner Größenordnung. Die
Bandbreite eines Monats beginnt bei diesem Betrag – sonst könnte das 95-%-Niveau unter
dem liegen, was schon feststeht. Derselbe Abruf wie für 5.2 liefert die Zahlen.

**Referenzklassen (6) – zurückgestellt.** Clockodo führt kein Feld, aus dem die Klasse
eines Projekts hervorgeht. Die Verteilung wird zunächst portfolioweit geschätzt; das
unterschätzt die Streuung zwischen Projekttypen, ist aber eine Näherung mit bekannter
Richtung statt einer geratenen Zuordnung.

**Kapazität (5.3).** Gerechnet wird in **Stunden**, nicht in Personentagen – eine
Taglänge ist nirgends hinterlegt. Die Sollarbeitszeit liegt vor (siehe oben), geplante
Abwesenheiten sind über `/v4/absences` erreichbar und noch nicht ausgewertet, der
Abschlag für ungeplante Abwesenheit ist eine Schätzgröße der Kalibrierung.

**Nächster Schritt** laut Spec Abschnitt 11: die Abrufquote-Verteilung schätzen, dann
Abwesenheiten, dann die Simulation.